# Parquet Basics — 10: End-to-End Performance Tuning

## What you will learn
This is the capstone notebook — combining all previous lessons into
a systematic performance tuning methodology.

In this notebook:
1. Profiling a slow Parquet pipeline — how to diagnose the problem
2. The tuning checklist — step-by-step process
3. Before/after benchmark — measuring improvement
4. Production Parquet configuration — recommended settings
5. Monitoring in Spark UI — what to look for
6. Summary of all optimization techniques learned


In [1]:
import os, time, pathlib, shutil, random, datetime, subprocess, glob as glib
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

MASTER   = 'spark://spark-master:7077'
DATA_DIR = '/workspace/data/parquet_basics'
pathlib.Path(DATA_DIR).mkdir(parents=True, exist_ok=True)

spark = (
    SparkSession.builder
    .appName('parquet-basics')
    .master(MASTER)
    .config('spark.executor.memory', '2g')
    .config('spark.driver.memory',   '1g')
    .config('spark.executor.cores',  '2')
    .config('spark.shuffle.sort.bypassMergeThreshold', '0')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print(f"Spark {spark.version} | DATA_DIR: {DATA_DIR}")

random.seed(42)
N = 200_000
CATEGORIES = ["Electronics","Clothing","Books","Food","Sports","Furniture"]
COUNTRIES  = ["US","UK","DE","FR","JP","CA","AU","BR"]
STATUSES   = ["pending","confirmed","shipped","delivered","cancelled"]

schema = StructType([
    StructField("order_id",    LongType()),
    StructField("customer_id", LongType()),
    StructField("product",     StringType()),
    StructField("category",    StringType()),
    StructField("country",     StringType()),
    StructField("quantity",    IntegerType()),
    StructField("price",       DoubleType()),
    StructField("discount",    DoubleType()),
    StructField("revenue",     DoubleType()),
    StructField("status",      StringType()),
    StructField("is_returned", BooleanType()),
    StructField("order_date",  DateType()),
])

base = datetime.date(2023, 1, 1)
rows = []
for i in range(N):
    qty  = random.randint(1, 20)
    price = round(random.uniform(5, 2000), 2)
    disc  = round(random.uniform(0, 0.4), 3)
    rev   = round(qty * price * (1 - disc), 2)
    d     = base + datetime.timedelta(days=random.randint(0, 365))
    rows.append((i+1, random.randint(1,50000),
                 f"Product_{random.randint(1,500)}",
                 random.choice(CATEGORIES), random.choice(COUNTRIES),
                 qty, price, disc, rev,
                 random.choice(STATUSES), random.random() < 0.05, d))

df = spark.createDataFrame(rows, schema)
df.cache().count()
print(f"Dataset: {N:,} rows | {len(schema)} columns")
df.printSchema()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/10 21:25:16 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark 4.0.2 | DATA_DIR: /workspace/data/parquet_basics


26/04/10 21:25:24 WARN TaskSetManager: Stage 0 contains a task of very large size (3057 KiB). The maximum recommended task size is 1000 KiB.
26/04/10 21:25:27 WARN TaskSetManager: Stage 1 contains a task of very large size (3057 KiB). The maximum recommended task size is 1000 KiB.


Dataset: 200,000 rows | 12 columns
root
 |-- order_id: long (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- country: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- price: double (nullable = true)
 |-- discount: double (nullable = true)
 |-- revenue: double (nullable = true)
 |-- status: string (nullable = true)
 |-- is_returned: boolean (nullable = true)
 |-- order_date: date (nullable = true)



## Step 1 — The Slow Baseline: A Poorly Configured Pipeline

We start with a poorly configured Parquet table and diagnose each issue.


In [2]:
# Write a deliberately bad Parquet setup:
# - No partitioning
# - Default (untuned) settings
# - No sorting
# - Many small files

SLOW_PATH = f"{DATA_DIR}/tuning/slow"

# Simulate: 50 small batches written over time
import shutil
shutil.rmtree(SLOW_PATH, ignore_errors=True)
pathlib.Path(SLOW_PATH).mkdir(parents=True, exist_ok=True)

batch_size = N // 50
for i in range(50):
    batch = df.limit(batch_size)
    batch.repartition(4).write.mode("append").parquet(SLOW_PATH)

import glob as glib
slow_files = glib.glob(f"{SLOW_PATH}/*.parquet")
slow_mb    = sum(pathlib.Path(f).stat().st_size for f in slow_files) / 1024/1024

print(f"Slow baseline:")
print(f"  Files      : {len(slow_files)}")
print(f"  Total size : {slow_mb:.1f} MB")
print(f"  Avg file   : {slow_mb/len(slow_files):.2f} MB")
print()
print("Problems:")
print("  ❌ Many small files (200) — high task overhead")
print("  ❌ No partitioning — full scan for every filtered query")
print("  ❌ Unordered data — poor min/max statistics, no skipping")
print("  ❌ Default snappy (ok) but no tuning for this workload")

26/04/10 21:25:27 WARN TaskSetManager: Stage 4 contains a task of very large size (3057 KiB). The maximum recommended task size is 1000 KiB.
26/04/10 21:25:29 WARN TaskSetManager: Stage 10 contains a task of very large size (3057 KiB). The maximum recommended task size is 1000 KiB.
26/04/10 21:25:31 WARN TaskSetManager: Stage 16 contains a task of very large size (3126 KiB). The maximum recommended task size is 1000 KiB.
26/04/10 21:25:32 WARN TaskSetManager: Stage 22 contains a task of very large size (3126 KiB). The maximum recommended task size is 1000 KiB.
26/04/10 21:25:33 WARN TaskSetManager: Stage 28 contains a task of very large size (3057 KiB). The maximum recommended task size is 1000 KiB.
26/04/10 21:25:34 WARN TaskSetManager: Stage 34 contains a task of very large size (3126 KiB). The maximum recommended task size is 1000 KiB.
26/04/10 21:25:35 WARN TaskSetManager: Stage 40 contains a task of very large size (3126 KiB). The maximum recommended task size is 1000 KiB.
26/04/1

Slow baseline:
  Files      : 200
  Total size : 6.4 MB
  Avg file   : 0.03 MB

Problems:
  ❌ Many small files (200) — high task overhead
  ❌ No partitioning — full scan for every filtered query
  ❌ Unordered data — poor min/max statistics, no skipping
  ❌ Default snappy (ok) but no tuning for this workload


## Step 2 — Diagnosing with explain() and Spark UI


In [3]:
# Diagnostic query
diagnostic = (spark.read.parquet(SLOW_PATH)
              .filter(F.col("category") == "Electronics")
              .filter(F.col("revenue") > 500)
              .groupBy("country").agg(F.sum("revenue")))

print("Physical plan of diagnostic query:")
diagnostic.explain(mode="formatted")
print()

# Run and time it
times = []
for _ in range(3):
    t0 = time.time()
    diagnostic.collect()
    times.append(time.time() - t0)
t_slow = sorted(times)[1]

print(f"Slow baseline query time: {t_slow:.3f}s")
print()
print("Observations from plan:")
print("  - No PartitionFilters (no partitioning)")
print("  - Filter node is ABOVE FileScan (pushed but no pruning)")
print("  - Reading ALL", len(slow_files), "files")

Physical plan of diagnostic query:
== Physical Plan ==
AdaptiveSparkPlan (7)
+- HashAggregate (6)
   +- Exchange (5)
      +- HashAggregate (4)
         +- Project (3)
            +- Filter (2)
               +- Scan parquet  (1)


(1) Scan parquet 
Output [3]: [category#15391, country#15392, revenue#15396]
Batched: true
Location: InMemoryFileIndex [file:/workspace/data/parquet_basics/tuning/slow]
PushedFilters: [IsNotNull(category), IsNotNull(revenue), EqualTo(category,Electronics), GreaterThan(revenue,500.0)]
ReadSchema: struct<category:string,country:string,revenue:double>

(2) Filter
Input [3]: [category#15391, country#15392, revenue#15396]
Condition : (((isnotnull(category#15391) AND isnotnull(revenue#15396)) AND (category#15391 = Electronics)) AND (revenue#15396 > 500.0))

(3) Project
Output [2]: [country#15392, revenue#15396]
Input [3]: [category#15391, country#15392, revenue#15396]

(4) HashAggregate
Input [2]: [country#15392, revenue#15396]
Keys [1]: [country#15392]
Functions 

[Stage 305:========================================>                (5 + 2) / 7]

Slow baseline query time: 0.071s

Observations from plan:
  - No PartitionFilters (no partitioning)
  - Filter node is ABOVE FileScan (pushed but no pruning)
  - Reading ALL 200 files


## Step 3 — Apply All Optimizations


In [4]:
FAST_PATH = f"{DATA_DIR}/tuning/fast"

print("Applying optimizations:")
print("  1. Compact: reduce 200 files to few large files")
print("  2. Partition by category (most common filter)")
print("  3. Sort within partitions by revenue (range filter)")
print("  4. zstd compression (better ratio)")
print("  5. Larger row groups (128 MB)")
print()

t0 = time.time()
(spark.read.parquet(SLOW_PATH)          # read all small files
      .repartitionByRange(2, "category", F.desc("revenue"))  # sort + partition
      .write
      .mode("overwrite")
      .partitionBy("category")           # directory-level partitioning
      .option("compression", "zstd")
      .option("parquet.block.size", str(128 * 1024 * 1024))
      .parquet(FAST_PATH))
t_optimize = time.time() - t0

fast_files = glib.glob(f"{FAST_PATH}/**/*.parquet", recursive=True)
fast_mb    = sum(pathlib.Path(f).stat().st_size for f in fast_files) / 1024/1024

print(f"Optimized result ({t_optimize:.1f}s to build):")
print(f"  Files      : {len(fast_files)}  (was {len(slow_files)})")
print(f"  Total size : {fast_mb:.1f} MB  (was {slow_mb:.1f} MB)")
print(f"  Avg file   : {fast_mb/max(len(fast_files),1):.1f} MB")

Applying optimizations:
  1. Compact: reduce 200 files to few large files
  2. Partition by category (most common filter)
  3. Sort within partitions by revenue (range filter)
  4. zstd compression (better ratio)
  5. Larger row groups (128 MB)



Optimized result (10.8s to build):
  Files      : 7  (was 200)
  Total size : 0.8 MB  (was 6.4 MB)
  Avg file   : 0.1 MB


## Step 4 — Before / After Benchmark


In [5]:
# Benchmark suite: same queries on slow vs fast path
benchmarks = [
    ("Category filter + agg",
     lambda p: spark.read.parquet(p).filter(F.col("category")=="Electronics").agg(F.sum("revenue")).collect()),
    ("Revenue range filter",
     lambda p: spark.read.parquet(p).filter((F.col("revenue")>500)&(F.col("revenue")<1000)).count()),
    ("Multi-filter + groupBy",
     lambda p: spark.read.parquet(p)
               .filter(F.col("category")=="Electronics")
               .filter(F.col("revenue")>500)
               .groupBy("country").agg(F.sum("revenue"),F.count("*")).collect()),
    ("Full scan aggregation",
     lambda p: spark.read.parquet(p).groupBy("category").agg(F.sum("revenue")).collect()),
    ("Column pruning (2 cols)",
     lambda p: spark.read.parquet(p).select("revenue","category").agg(F.sum("revenue")).collect()),
]

print(f"{'Query':<35} {'Slow':>8} {'Fast':>8} {'Speedup':>10}")
print("-" * 65)

total_slow, total_fast = 0, 0
for label, fn in benchmarks:
    ts, tf = [], []
    for _ in range(3):
        t0=time.time(); fn(SLOW_PATH); ts.append(time.time()-t0)
        t0=time.time(); fn(FAST_PATH); tf.append(time.time()-t0)
    t_s, t_f = sorted(ts)[1], sorted(tf)[1]
    total_slow += t_s; total_fast += t_f
    print(f"  {label:<33} {t_s:>6.3f}s {t_f:>6.3f}s {t_s/t_f:>9.1f}x")

print("-" * 65)
print(f"  {'TOTAL':<33} {total_slow:>6.3f}s {total_fast:>6.3f}s {total_slow/total_fast:>9.1f}x")

Query                                   Slow     Fast    Speedup
-----------------------------------------------------------------


  Category filter + agg              6.097s  0.513s      11.9x


  Revenue range filter               6.107s  0.571s      10.7x


  Multi-filter + groupBy             6.111s  0.567s      10.8x


  Full scan aggregation              5.932s  0.573s      10.3x


  Column pruning (2 cols)            5.884s  0.533s      11.0x
-----------------------------------------------------------------
  TOTAL                             30.131s  2.759s      10.9x


## Step 5 — Production Configuration Template


In [6]:
print("""
# ═══════════════════════════════════════════════════════════
# Production Parquet Configuration Template
# ═══════════════════════════════════════════════════════════

# ─── Reading ────────────────────────────────────────────────
df = (
    spark.read
         .schema(explicit_schema)          # ALWAYS explicit schema
         .option("mergeSchema", False)     # True only if files have different schemas
         .parquet("/data/table/")
         .select("col1", "col2", "col3")  # column pruning — specify what you need
         .filter(col("partition_col") == value)  # partition filter first
         .filter(col("other_col") > threshold)   # then row-level filters
)

# ─── Writing ────────────────────────────────────────────────
(df
 .repartitionByRange(n_files, "partition_col", col("sort_col").desc())
 .write
 .mode("overwrite")
 .partitionBy("partition_col")            # directory-level partitioning
 .option("compression",        "zstd")   # or "snappy" for hot tables
 .option("parquet.block.size", str(128 * 1024 * 1024))  # 128 MB row groups
 .option("parquet.page.size",  str(1   * 1024 * 1024))  # 1 MB pages (default)
 .parquet("/data/output/")
)

# ─── Partitioning column selection ──────────────────────────
# ✅ Good: date, year/month, category (< 1000 distinct values)
# ❌ Bad:  user_id, order_id, UUID (too many distinct values)

# ─── File count target ──────────────────────────────────────
# Target: 128 MB – 1 GB per file
# Formula: n_files = max(1, int(total_data_gb * 1000 / 128))

# ─── Dynamic partition overwrite ────────────────────────────
# Replace only affected partitions (not entire table)
spark.conf.set("spark.sql.sources.partitionOverwriteMode", "dynamic")
""")


# ═══════════════════════════════════════════════════════════
# Production Parquet Configuration Template
# ═══════════════════════════════════════════════════════════

# ─── Reading ────────────────────────────────────────────────
df = (
    spark.read
         .schema(explicit_schema)          # ALWAYS explicit schema
         .option("mergeSchema", False)     # True only if files have different schemas
         .parquet("/data/table/")
         .select("col1", "col2", "col3")  # column pruning — specify what you need
         .filter(col("partition_col") == value)  # partition filter first
         .filter(col("other_col") > threshold)   # then row-level filters
)

# ─── Writing ────────────────────────────────────────────────
(df
 .repartitionByRange(n_files, "partition_col", col("sort_col").desc())
 .write
 .mode("overwrite")
 .partitionBy("partition_col")            # directory-level partitioning
 .option("compression",        "zstd")   # or "snappy" for hot tables
 .option("par

## Step 6 — Complete Optimization Checklist

A systematic checklist for every Parquet table you create or optimize.


In [7]:
print("""
Parquet Performance Checklist
══════════════════════════════════════════════════════════════

SCHEMA
  ☐ Define explicit schema (never inferSchema for production)
  ☐ Use appropriate types (Int not Long for small ints, etc.)
  ☐ Mark not-nullable columns as nullable=False

WRITE TIME
  ☐ Choose partition column (< 1000 values, commonly filtered)
  ☐ Sort within partitions by most selective filter column
  ☐ Choose compression: zstd (storage) / snappy (hot) / lz4 (temp)
  ☐ Target 128 MB – 1 GB per output file
  ☐ Use repartitionByRange() for sorted+partitioned writes

READ TIME
  ☐ Use select() to specify only needed columns
  ☐ Apply partition filter FIRST (partition pruning)
  ☐ Apply row-level filters on original columns (not derived)
  ☐ Check explain() for PushedFilters and PartitionFilters

MAINTENANCE
  ☐ Monitor file count (alert if > 10K files in one table)
  ☐ Compact small files after streaming writes
  ☐ Use Delta/Iceberg OPTIMIZE for managed tables

SPARK SETTINGS
  ☐ AQE enabled (default in Spark 3.0+)
  ☐ AQE coalescing enabled (merges small post-shuffle files)
  ☐ autoBroadcastJoinThreshold set appropriately

MONITORING (Spark UI)
  ☐ Check Stages: is one task much slower? (data skew)
  ☐ Check SQL: are filters showing as PushedFilters?
  ☐ Check Storage: cache hit rate for repeated queries
  ☐ Check I/O: bytes read vs bytes written (compression working?)
""")


Parquet Performance Checklist
══════════════════════════════════════════════════════════════

SCHEMA
  ☐ Define explicit schema (never inferSchema for production)
  ☐ Use appropriate types (Int not Long for small ints, etc.)
  ☐ Mark not-nullable columns as nullable=False

WRITE TIME
  ☐ Choose partition column (< 1000 values, commonly filtered)
  ☐ Sort within partitions by most selective filter column
  ☐ Choose compression: zstd (storage) / snappy (hot) / lz4 (temp)
  ☐ Target 128 MB – 1 GB per output file
  ☐ Use repartitionByRange() for sorted+partitioned writes

READ TIME
  ☐ Use select() to specify only needed columns
  ☐ Apply partition filter FIRST (partition pruning)
  ☐ Apply row-level filters on original columns (not derived)
  ☐ Check explain() for PushedFilters and PartitionFilters

MAINTENANCE
  ☐ Monitor file count (alert if > 10K files in one table)
  ☐ Compact small files after streaming writes
  ☐ Use Delta/Iceberg OPTIMIZE for managed tables

SPARK SETTINGS
  ☐ AQE